In [1]:
import pandas as pd
import numpy as np
import sklearn 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

In [2]:
# 1. Call the function to fetch the data object
housing = fetch_california_housing(as_frame = True) # as_frame for direct DataFrame
X = housing.data
Y = housing.target


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state= 456 # last 3 id digits
)

In [4]:
scale = StandardScaler()

# Fit the scaler ONLY on the training data, then transform both sets
X_train_scaled = pd.DataFrame(
    scale.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scale.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

In [5]:
X_train_scaled.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
1022,-0.746693,-0.922976,1.432652,1.590760,-0.957080,-0.075436,1.435423,-0.108500
7945,0.282415,1.225602,-0.084085,-0.224180,-0.386435,-0.000879,-0.823579,0.711551
3980,0.239169,0.429832,0.048203,-0.197956,-0.929738,-0.016605,-0.673604,0.476536
10689,-0.530571,-0.763822,-0.529645,-0.280413,-0.933266,-0.154423,-0.940747,0.921564
11510,0.419298,0.191101,-0.479132,0.017933,-0.758633,-0.110601,-0.884507,0.731552


In [37]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 1. Initialize the baseline model
# hidden_layer_sizes=(64, 32) creates two hidden layers
# activation='relu' fulfills the non-linear requirement
# solver='adam' is a robust optimizer for regression
baseline_model = MLPRegressor(
    hidden_layer_sizes=(64, 32), # Typical neuron layering size, BUT will explore this deeper
    activation='relu', # As mentioned in the briefing
    solver='adam', # Best solver for dataset(>10K)
    max_iter=500, # Not too high to sav time
    random_state= 143
)

# 2. Train the model on the scaled training data
print("Training the baseline model...")
baseline_model.fit(X_train_scaled, y_train)

# 3. Evaluate on the test set to get our baseline performance
# We use the test set here just to see how the baseline performs [cite: 62, 64]
y_pred_baseline = baseline_model.predict(X_test_scaled)
baseline_mse = mean_squared_error(y_test, y_pred_baseline)

print(f"Baseline Test Mean Squared Error: {baseline_mse:.4f}")

Training the baseline model...
Baseline Test Mean Squared Error: 0.2569


# however we can expand on this deeper by finding the most optimal paramaters, for this model
## to do this, we are gonna use GRIDSEARCH to iterate the listed outcomes

In [36]:
from sklearn.model_selection import GridSearchCV

# Define a dictionary of parameters to test
param_grid = {
    'hidden_layer_sizes': [(64, 32), (100, 50), (32, 16)],
    'alpha': [0.0001, 0.001, 0.01], # L2 regularization penalty to prevent overfitting
    'learning_rate_init': [0.001, 0.01]
}

# Initialize a base model
base_mlp = MLPRegressor(activation='relu', solver='adam', random_state=143, max_iter=500) #This part is the same always

# Run the grid search 
grid_search = GridSearchCV(
    base_mlp, 
    param_grid, 
    cv=3, # 3-fold cross-validation splits data into 3 then averages, ex.(1,2 -> 3 , 1,3 -> 2 then 2,3 -> 1)
    scoring='neg_mean_squared_error', # target value
    n_jobs=-1, # Speeds up the process
    verbose = 1 # gives updates
)

print("Starting grid search...")
grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters found: {grid_search.best_params_}")

best_baseline_model = grid_search.best_estimator_
y_pred_best_baseline = best_baseline_model.predict(X_test_scaled)
best_baseline_mse = mean_squared_error(y_test, y_pred_best_baseline)

print(f"Optimized Baseline Test Mean Squared Error: {best_baseline_mse:.4f}")

Starting grid search...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best parameters found: {'alpha': 0.01, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
Optimized Baseline Test Mean Squared Error: 0.2616
